In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_113_Shadipur_Delhi_CPCB_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,159.49,336.99,25.51,42.48,38.36,30.65,18.53,0.97,19.64,...,4.32,NaN,85.45,0.51,239.25,NaN,NaN,74.96,NaN,0.26
1,2024-01-02,201.22,347.06,25.10,43.58,38.94,34.64,16.85,0.64,18.94,...,4.31,NaN,80.93,0.50,242.21,NaN,NaN,81.38,NaN,0.26
2,2024-01-03,235.04,359.00,17.05,37.73,32.58,42.14,16.19,1.35,16.04,...,4.26,NaN,92.29,0.45,184.84,NaN,NaN,64.66,NaN,0.26
3,2024-01-04,223.36,386.80,16.18,35.34,31.05,42.19,10.31,1.44,14.90,...,4.31,NaN,97.32,0.48,277.63,NaN,NaN,56.41,NaN,0.27
4,2024-01-05,153.35,329.02,17.33,33.56,31.10,46.05,12.54,1.19,22.23,...,4.28,NaN,NaN,0.38,283.75,NaN,NaN,48.90,NaN,0.27
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,126.25,177.30,16.22,73.89,50.69,39.68,8.16,1.06,43.30,...,2.17,NaN,87.55,0.68,89.36,NaN,NaN,46.85,NaN,0.22
362,2024-12-28,97.93,132.89,16.63,68.74,48.97,29.73,9.73,1.06,45.46,...,2.17,NaN,94.55,0.27,207.88,NaN,NaN,77.86,NaN,0.23
363,2024-12-29,80.51,104.45,16.02,61.70,44.40,16.59,8.53,1.06,22.44,...,2.21,NaN,90.73,0.65,290.45,NaN,NaN,204.67,NaN,0.23
364,2024-12-30,73.63,102.85,14.30,60.02,42.26,12.85,9.50,1.05,24.09,...,2.20,NaN,83.16,0.57,278.28,NaN,NaN,144.00,NaN,0.23


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 21)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Xylene (µg/m³)', 'BP (mmHg)']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 Timestamp              0
PM2.5 (µg/m³)          0
PM10 (µg/m³)           0
NO (µg/m³)             0
NO2 (µg/m³)            0
NOx (ppb)              0
NH3 (µg/m³)            0
SO2 (µg/m³)            0
CO (mg/m³)             0
Ozone (µg/m³)          0
Benzene (µg/m³)        0
Toluene (µg/m³)        0
Eth-Benzene (µg/m³)    0
MP-Xylene (µg/m³)      0
RH (%)                 0
WS (m/s)               0
WD (deg)               0
SR (W/mt2)             0
VWS (m/s)              0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (366, 19)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         159.49        336.99       25.51        42.48   
1  2024-01-02         201.22        347.06       25.10        43.58   
2  2024-01-03         235.04        359.00       17.05        37.73   
3  2024-01-04         223.36        386.80       16.18        35.34   
4  2024-01-05         153.35        329.02       17.33        33.56   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  \
0      38.36        30.65        18.53        0.97          19.64   
1      38.94        34.64        16.85        0.64          18.94   
2      32.58        42.14        16.19        1.35          16.04   
3      31.05        42.19        10.31        1.44          14.90   
4      31.10        46.05        12.54        1.19          22.23   

   Benzene (µg/m³)  Toluene (µg/m³)  Eth-Benzene (µg/m³)  MP-Xylene (µg/m³)  \
0             6.16             8.21                 4.91

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),Toluene (µg/m³),Eth-Benzene (µg/m³),MP-Xylene (µg/m³),RH (%),WS (m/s),WD (deg),SR (W/mt2),VWS (m/s)
0,2024-01-01,0.532372,0.614020,0.486218,-0.574360,-0.421141,-0.848899,1.676944,-0.328715,-0.767675,2.556719,2.339230,2.339515,2.059495,1.313202,0.378975,0.904028,-1.574351,1.380091
1,2024-01-02,1.109572,0.686938,0.428849,-0.495265,-0.371736,-0.458697,1.126216,-1.402908,-0.840462,2.530503,2.393143,2.318277,2.048159,1.108430,0.326850,0.954273,-1.478738,1.380091
2,2024-01-03,1.577363,0.773397,-0.697552,-0.915906,-0.913494,0.274767,0.909859,0.908235,-1.142010,2.547980,2.305535,2.286421,1.991476,1.623077,0.066224,-0.019558,-1.727748,1.380091
3,2024-01-04,1.415808,0.974700,-0.819287,-1.087758,-1.043822,0.279657,-1.017688,1.201197,-1.260549,2.513025,2.332491,2.233327,2.048159,1.850953,0.222600,1.555512,-1.850615,1.468213
4,2024-01-05,0.447445,0.556308,-0.658373,-1.215748,-1.039563,0.657146,-0.286663,0.387414,-0.498361,2.495548,2.366186,2.328896,2.014149,-0.044995,-0.298651,1.659396,-1.962461,1.468213
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,0.072604,-0.542315,-0.813690,1.684160,0.629153,0.034191,-1.722488,-0.035753,1.692541,-0.641681,-0.349669,-0.453211,-0.377852,1.408339,1.265101,-1.640290,-1.992991,1.027604
362,2024-12-28,-0.319111,-0.863894,-0.756321,1.313852,0.482640,-0.938871,-1.207820,-0.035753,1.917142,-0.650420,-0.403582,-0.516923,-0.377852,1.725463,-0.872027,0.371536,-1.531161,1.115726
363,2024-12-29,-0.560061,-1.069831,-0.841675,0.807645,0.093358,-2.223899,-1.601197,-0.035753,-0.476525,-0.606726,-0.336191,-0.485067,-0.332506,1.552404,1.108726,1.773126,0.357413,1.115726
364,2024-12-30,-0.655223,-1.081417,-1.082347,0.686845,-0.088931,-2.589652,-1.283218,-0.068304,-0.304954,-0.624203,-0.363147,-0.453211,-0.343843,1.209457,0.691725,1.566545,-0.546142,1.115726


In [10]:
df.to_excel('shadipur2024.xlsx', index=False)